# Объединение ретриверов и геопереранжирование

Каждый источник ищет top-N → объединяем по `item_id` → выбираем итоговые 50.
Сравниваем N = 50, 500, 5000 и пять вариантов: RRF, только расстояние,
RRF + локация, RRF + расстояние, RRF + оба геосигнала.

Источники: BM25 query–title, query–description, infm–infm; Giga и BGE query–title.
Сырые scores разных моделей не складываем. RRF складывает `1 / (60 + rank)`;
сумму нормируем на максимум внутри запроса. Отсутствие документа у источника даёт 0.
Пустой `search_infm_params_text` не добавляет кандидатов от BM25 infm.

Сохраняем top-5000 каждого источника батчами: меньшие N получаем срезом.
Dense-индексы **загружаются из эксперимента по заголовкам**; корпус заново не кодируется.
Энкодеры всё ещё нужны для запросов. Модели запускаются последовательно и освобождаются.

Сплит — прежний `split_queries(seed=42)`: 70/15/15 по нормализованным запросам.
Корпус общий; центры локаций считаются только по train. Набор источников и N выбираем
на val, затем считаем test для зафиксированных конфигураций. Геовеса ниже — стартовые
значения, их тоже можно менять по val. Test не используем для подбора.

In [ ]:
from pathlib import Path

REPO_PATH = None  # путь к обновлённому репозиторию; None — поиск рядом с ноутбуком
DATA_PATH = None  # train.parquet; None — repo/data или /kaggle/input
DENSE_INDEX_DIR = None  # папка с giga/ и bge/; локально repo/dense_indexes, Kaggle /kaggle/working/dense_indexes
INSTALL_DEPENDENCIES = False  # True для нового окружения Kaggle
DENSE_MODELS = ["giga", "bge"]  # [] — быстрый эксперимент только с BM25
DEVICE = None  # автоматический выбор CPU/CUDA

SEED = 42
N_QUERIES = 500  # на каждую из val/test; None — все
NS = [50, 500, 5000]
K = 50
QUERY_BATCH_SIZE = 16  # ограничивает память при поиске и объединении
DENSE_BATCH_SIZE = 16
BM25_LEMMATIZE = True  # лемматизация корпуса и запросов для всех BM25-источников
LOCATION_WEIGHT = 0.2
DISTANCE_WEIGHT = 0.2
DISTANCE_SCALE_KM = 100

## Окружение

Локально: `uv sync --extra dense`. На Kaggle укажите путь к сохранённым индексам
(можно прикрепить их как Dataset) и обновлённому коду проекта. Если индексов нет,
ячейка проверки остановится до построения BM25; автоматического запуска dense fit нет.

In [ ]:
import sys
import subprocess

if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "bm25s>=0.3.11", "pymorphy3>=2", "razdel>=0.5",
        "polars>=1.44,<2", "pandas>=2", "scikit-learn>=1.6,<2",
        "faiss-cpu>=1.10,<2", "sentence-transformers>=5.1,<6", "transformers>=4.51,<5",
    ])

roots = [Path.cwd(), *Path.cwd().parents, *Path("/kaggle/working").glob("*")]
ROOT = Path(REPO_PATH) if REPO_PATH else next((p for p in roots if (p / "src/retrievers/hybrid.py").is_file()), None)
if ROOT is None or not (ROOT / "src/retrievers/hybrid.py").is_file():
    raise FileNotFoundError("Укажите REPO_PATH на обновлённый код с src/retrievers/hybrid.py.")
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import gc
import hashlib
import json
from time import perf_counter
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from src.split import split_queries
from src.retrievers.bm25 import BM25Retriever
from src.retrievers.hybrid import HybridRetriever

if DATA_PATH is None:
    matches = [p for p in [ROOT / "data/train.parquet", *Path("/kaggle/input").rglob("train.parquet")] if p.is_file()]
    if len(matches) != 1:
        raise ValueError(f"Укажите DATA_PATH: найдено {len(matches)} файлов train.parquet.")
    DATA_PATH = matches[0]
DATA_PATH = Path(DATA_PATH)
if DENSE_INDEX_DIR is None:
    DENSE_INDEX_DIR = Path("/kaggle/working/dense_indexes") if Path("/kaggle/working").is_dir() else ROOT / "dense_indexes"
DENSE_INDEX_DIR = Path(DENSE_INDEX_DIR)
ARTIFACTS = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else ROOT) / "artifacts/hybrid"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
assert 1 <= K <= 50 and all(n >= K for n in NS)
assert QUERY_BATCH_SIZE > 0

# Проверяем оба индекса до дорогой части эксперимента.
dense_metadata = {}
for name in DENSE_MODELS:
    path = DENSE_INDEX_DIR / name
    if not all((path / f).is_file() for f in ("index.faiss", "metadata.json")):
        raise FileNotFoundError(f"Нет сохранённого индекса {path}. Укажите DENSE_INDEX_DIR или DENSE_MODELS=[].")
    raw = (path / "metadata.json").read_bytes()
    metadata = json.loads(raw)
    params = metadata["params"]
    if (params["model"] != name or params["item_text_column"] != "item_title_raw"
            or params["query_text_column"] != "search_query"):
        raise ValueError(f"{path}: нужен индекс {name} для search_query → item_title_raw.")
    dense_metadata[name] = {"digest": hashlib.sha256(raw).hexdigest(), "params": params}
    del metadata, raw
print("Данные:", DATA_PATH, "\nИндексы:", DENSE_INDEX_DIR, "\nАртефакты:", ARTIFACTS)

## Данные и центры локаций

Для каждого item_id берём последнее объявление в исходном файле — как в dense-ноутбуке.
Запрос — полный контекст из пяти полей. Центры считаются только по train:
медианы координат по item_location_id; если такого центра нет, медианы координат
связанных объявлений по search_location_id, без повторов пары (локация, item_id).
center_source и center_items показывают происхождение и число опорных объявлений.
Это оценка расположения локации, а не координаты пользователя. При неизвестном
центре геобонус равен 0; чистая сортировка по расстоянию использует RRF для ничьих.

In [ ]:
from src.geo import location_centers

QUERY_COLS = ["search_query", "search_location_id", "search_is_delivery_search", "search_infm_params_text", "search_category"]
GEO_COLS = ["item_location_id", "item_latitude", "item_longitude"]
data = pl.read_parquet(DATA_PATH, columns=QUERY_COLS + ["item_id"] + GEO_COLS)
data = data.with_columns(
    pl.Series("split", split_queries(data["search_query"], seed=SEED)),
    pl.col("item_latitude", "item_longitude").cast(pl.Float64),
)
items = pd.DataFrame(
    data.select("item_id", *GEO_COLS).unique(subset="item_id", keep="last", maintain_order=True)
    .sort("item_id").to_dict(as_series=False)
)
train_items = data.filter(pl.col("split") == "train").select("search_location_id", "item_id", *GEO_COLS)
centers = location_centers(pd.DataFrame(train_items.to_dict(as_series=False)))
queries, targets = {}, {}
for split in ("val", "test"):
    part = (data.filter(pl.col("split") == split).group_by(QUERY_COLS)
            .agg(pl.col("item_id").unique().alias("targets")).sort(QUERY_COLS))
    if N_QUERIES is not None:
        part = part.sample(n=min(N_QUERIES, part.height), seed=SEED, shuffle=True)
    queries[split] = pd.DataFrame(part.select(QUERY_COLS).to_dict(as_series=False)).join(centers, on="search_location_id")
    targets[split] = [set(ids) for ids in part["targets"].to_list()]
    assert all(targets[split])
    assert set().union(*targets[split]).issubset(set(items.item_id))
    print(split, "контекстов:", len(part), "без центра:", queries[split].center_lat.isna().sum())
display(data.group_by("split").len().sort("split"))
del data, train_items, part
gc.collect()

# Длинные тексты читаем только при построении соответствующего BM25-индекса.
def read_item_text(column):
    frame = (pl.scan_parquet(DATA_PATH).select("item_id", column)
             .unique(subset="item_id", keep="last", maintain_order=True).sort("item_id").collect())
    return pd.DataFrame(frame.to_dict(as_series=False))

## Один проход каждого источника: кеш top-5000

Индексы и энкодеры держим в памяти по одному. Батчевые parquet содержат локальный
`query_index`: позицию внутри соответствующего батча. Отпечаток кеша учитывает
запросы и их порядок, файл данных (путь, размер, mtime), параметры источника,
код ретривера, размер батча и глубину поиска. Геовеса в него не входят.
Готовые батчи повторно не считаются; прерванный прогон можно продолжить.
При ручной замене содержимого данных/индекса с сохранением метаданных удалите соответствующий кеш.

In [ ]:
sources = {
    "bm25_title": {"kind": "bm25", "item_text_column": "item_title_raw", "query_text_column": "search_query"},
    "bm25_description": {"kind": "bm25", "item_text_column": "item_description_raw", "query_text_column": "search_query"},
    "bm25_infm": {"kind": "bm25", "item_text_column": "item_infm_params_text", "query_text_column": "search_infm_params_text"},
}
for spec in sources.values():
    spec.update(lemmatize=BM25_LEMMATIZE, k1=1.5, b=0.75)
sources.update({name: {"kind": "dense", **dense_metadata[name]} for name in DENSE_MODELS})
stat = DATA_PATH.stat()
query_hashes = {split: hashlib.sha256(pd.util.hash_pandas_object(q, index=True).to_numpy().tobytes()).hexdigest()
                for split, q in queries.items()}
cache_dirs = {}
for name, spec in sources.items():
    code_path = ROOT / "src/retrievers" / ("bm25.py" if spec["kind"] == "bm25" else "dense.py")
    signature = {"source": spec, "queries": query_hashes, "max_n": max(NS), "batch": QUERY_BATCH_SIZE,
                 "data": [str(DATA_PATH.resolve()), stat.st_size, stat.st_mtime_ns],
                 "code": hashlib.sha256(code_path.read_bytes()).hexdigest()}
    if spec["kind"] == "dense":
        index_stat = (DENSE_INDEX_DIR / name / "index.faiss").stat()
        signature["index"] = [index_stat.st_size, index_stat.st_mtime_ns]
    digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
    cache_dirs[name] = ARTIFACTS / "candidates" / name / digest
    cache_dirs[name].mkdir(parents=True, exist_ok=True)
    (cache_dirs[name] / "manifest.json").write_text(json.dumps(signature, ensure_ascii=False, indent=2))

for name, spec in sources.items():
    jobs = [(split, start, cache_dirs[name] / split / f"{start:07d}.parquet")
            for split, q in queries.items() for start in range(0, len(q), QUERY_BATCH_SIZE)]
    pending = [(split, start, path) for split, start, path in jobs if not path.is_file()]
    if not pending:
        print(name, "— все батчи уже в кеше")
        continue
    started = perf_counter()
    if spec["kind"] == "bm25":
        corpus = read_item_text(spec["item_text_column"])
        model = BM25Retriever(**{key: value for key, value in spec.items() if key != "kind"},
                              batch_size=QUERY_BATCH_SIZE).fit(corpus)
    else:
        from src.retrievers.dense import DenseRetriever
        corpus = read_item_text("item_title_raw")
        model = DenseRetriever.load(DENSE_INDEX_DIR / name, device=DEVICE)
        expected_ids = corpus.loc[corpus.item_title_raw.fillna("").str.strip().ne(""), "item_id"].tolist()
        if model.item_ids_.tolist() != expected_ids:
            raise ValueError(f"{name}: item_id/порядок корпуса не соответствует сохранённому индексу.")
        model.batch_size = DENSE_BATCH_SIZE
        del expected_ids
    del corpus
    print(name, "— индекс готов за", round(perf_counter() - started, 1), "с")
    for i, (split, start, path) in enumerate(pending, 1):
        batch = queries[split].iloc[start:start + QUERY_BATCH_SIZE]
        result = model.retrieve(batch, k=max(NS))
        assert result.query_index.between(0, len(batch) - 1).all()
        assert not result.duplicated(["query_index", "item_id"]).any()
        assert np.isfinite(result.score.to_numpy(dtype=float)).all()
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".tmp.parquet")
        pl.DataFrame(result.to_dict(orient="list"), schema={"query_index": pl.Int64, "item_id": pl.String,
                                                          "rank": pl.Int64, "score": pl.Float64}).write_parquet(temporary)
        temporary.replace(path)
        if i % 10 == 0 or i == len(pending):
            print(name, f"{i}/{len(pending)} батчей; {perf_counter() - started:.1f} с")
    del model, result
    gc.collect()
    if spec["kind"] == "dense":
        import torch
        torch.cuda.empty_cache()

## Объединение, метрики и варианты

`HybridRetriever` реализует общий интерфейс `Retriever`. Для обычного использования:
```python
hybrid = HybridRetriever({"bm25": fitted_bm25, "giga": loaded_giga},
                         n_candidates=500, location_weight=0.2, distance_weight=0.2).fit(items)
result = hybrid.retrieve(queries_with_centers, k=50)
```
`fit` обёртки сохраняет только геометаданные. Для эксперимента ниже используем
`combine` и `rerank` на сохранённых выдачах, чтобы не держать все модели в памяти.

Для каждого N считаем:
- `pool_recall`: доля релевантных объявлений во всём union, без ограничения в 50;
- `oracle_recall@50`: максимум Recall@50 при идеальном порядке данного union;
- `recall@50`: фактическая выдача после переранжирования;
- `mean_pool_size`: среднее число уникальных кандидатов после объединения.

Во всех метриках участвуют и запросы без кандидатов (их recall равен 0).
Сначала геосигналы оцениваются на **всём union**, затем выбирается top-50.

In [ ]:
source_sets = {"bm25_title": ["bm25_title"], "bm25_all": ["bm25_title", "bm25_description", "bm25_infm"]}
if DENSE_MODELS:
    source_sets["all"] = list(sources)
variants = {
    "rrf": dict(location_weight=0, distance_weight=0, distance_only=False),
    "distance_only": dict(location_weight=0, distance_weight=0, distance_only=True),
    "rrf_location": dict(location_weight=LOCATION_WEIGHT, distance_weight=0, distance_only=False),
    "rrf_distance": dict(location_weight=0, distance_weight=DISTANCE_WEIGHT, distance_only=False),
    "rrf_both": dict(location_weight=LOCATION_WEIGHT, distance_weight=DISTANCE_WEIGHT, distance_only=False),
}
configurations = pd.DataFrame([
    {"sources": group, "N": n, "variant": variant, "distance_scale_km": DISTANCE_SCALE_KM, **params}
    for group in source_sets for n in NS for variant, params in variants.items()
])
PARAMS = ["location_weight", "distance_weight", "distance_scale_km", "distance_only"]
CONFIG_COLS = ["sources", "N", "variant", *PARAMS]


def recall_per_query(result, relevant):
    found = result.groupby("query_index").item_id.agg(set).to_dict()
    return np.array([len(ids & found.get(i, set())) / len(ids) for i, ids in enumerate(relevant)])


def evaluate(split, configs):
    configs = configs.reset_index(drop=True)
    totals = np.zeros((len(configs), 4))
    needed = set().union(*(source_sets[group] for group in configs.sources))
    hybrid = HybridRetriever().fit(items)
    q = queries[split]
    for start in range(0, len(q), QUERY_BATCH_SIZE):
        batch = q.iloc[start:start + QUERY_BATCH_SIZE]
        relevant = targets[split][start:start + len(batch)]
        denominators = np.array([len(ids) for ids in relevant])
        outputs = {name: pd.DataFrame(pl.read_parquet(cache_dirs[name] / split / f"{start:07d}.parquet").to_dict(as_series=False))
                   for name in needed}
        for (group, n), subset in configs.groupby(["sources", "N"], sort=False):
            hybrid.set_params(n_candidates=int(n))
            features = hybrid.combine(batch, {name: outputs[name] for name in source_sets[group]})
            hits = [item_id in relevant[i] for i, item_id in features[["query_index", "item_id"]].itertuples(index=False, name=None)]
            positions = features.query_index.to_numpy(dtype=int)
            found = np.bincount(positions, weights=hits, minlength=len(batch))
            pool_size = np.bincount(positions, minlength=len(batch))
            coverage = (found / denominators).sum()
            oracle = (np.minimum(found, K) / denominators).sum()
            for config_i, row in subset.iterrows():
                hybrid.set_params(**{p: row[p] for p in PARAMS})
                result = hybrid.rerank(features, k=K)
                totals[config_i] += [recall_per_query(result, relevant).sum(), coverage, oracle, pool_size.sum()]
        print(split, f"{min(start + len(batch), len(q))}/{len(q)} запросов", end="\r")
    print()
    result = configs.copy()
    result[[f"recall@{K}", "pool_recall", f"oracle_recall@{K}", "mean_pool_size"]] = totals / len(q)
    return result

## Validation: выбираем N и набор источников

Таблица содержит все заранее заданные варианты. Для каждой стратегии фиксируем
лучшую конфигурацию по val; при равенстве берём меньший N. Также фиксируем одного
общего победителя **до просмотра test**. Значение RRF для одного источника монотонно
его исходному рангу, поэтому `bm25_title / rrf` — контроль обычного BM25.

In [ ]:
validation = evaluate("val", configurations)
validation.to_csv(ARTIFACTS / "validation.csv", index=False)
display(validation.pivot_table(index=["sources", "N"], columns="variant", values=f"recall@{K}"))
display(validation[["sources", "N", "pool_recall", f"oracle_recall@{K}", "mean_pool_size"]].drop_duplicates())
ordered = validation.sort_values([f"recall@{K}", "N", "sources", "variant"], ascending=[False, True, True, True])
best = ordered.drop_duplicates("variant").copy()
best.to_csv(ARTIFACTS / "selected_on_val.csv", index=False)
winner = ordered.iloc[0]
print("Выбран на val:", winner[CONFIG_COLS].to_dict())
display(best)

## Test: только зафиксированные конфигурации

Сравниваем выбранные на val конфигурации пяти стратегий и контрольный BM25-title.
Общий победитель остаётся выбранным по val. При большой `pool_recall`, но слабой
`recall@50`, проблема в порядке кандидатов; при низкой полноте union — в источниках/глубине поиска.

In [ ]:
control = configurations.query("sources == 'bm25_title' and variant == 'rrf'").sort_values("N").head(1)
test_configs = pd.concat([best[CONFIG_COLS], control], ignore_index=True).drop_duplicates()
comparison = evaluate("test", test_configs).merge(
    validation[CONFIG_COLS + [f"recall@{K}"]].rename(columns={f"recall@{K}": f"val_recall@{K}"}),
    on=CONFIG_COLS, validate="one_to_one",
).rename(columns={f"recall@{K}": f"test_recall@{K}"})
comparison["selected_on_val"] = comparison[CONFIG_COLS].eq(winner[CONFIG_COLS]).all(axis=1)
comparison.to_csv(ARTIFACTS / "test.csv", index=False)
display(comparison)
print("Результаты и кеш:", ARTIFACTS)